# Demo — Fine-Grained Access with Lake Formation (live AWS)

**Trailhead Provisions** must let a marketing analyst work with customer data **without seeing
personal or contact PII (`full_name`, `email`, `phone`)**. This walkthrough demonstrates the
*technique*: make a **column-level Lake Formation grant**, **verify it by querying as that persona
in Athena**, and inspect how the underlying **S3 → Glue → Lake Formation → IAM** governance chain
is declared in Terraform. The exercise asks you to configure all four personas, run the full
verification, and explain the Terraform module.

> Runs against **live AWS Glue, Lake Formation, and Amazon Athena** (no local fallback).

## Step 0: Connect to your Cloud Lab
Run the cell below to configure your live AWS environment and ensure the `trailhead` Glue Data Catalog, S3 tables, and Lake Formation enforcement are seeded.

In [ ]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog, S3 tables, and Lake Formation enforcement if needed
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend, verify_step7, PERSONAS

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__}")
print(f"  Glue DB:          {gc.database}")
print(f"  Region:           {gc.region}")
print(f"  Athena workgroup: {gc.athena_workgroup}")
print(f"  Results bucket:   {gc.results_bucket}")
print("Personas:", list(PERSONAS))
gc.reset()
gc.grants_summary()

## Grant column-level access
The analyst should see the `customer` table **without** the `full_name`, `email`, and `phone` PII columns. Under the live backend this is a real Lake Formation column grant.

In [ ]:
gc.grant_columns("co_marketing_analyst", "customer", exclude=["full_name", "email", "phone"])
gc.grants_summary()

## Verify by querying as the persona
`query_as` assumes the persona role via STS and runs the query through Athena (live) so Lake Formation enforcement is what you see.

In [ ]:
df = gc.query_as("co_marketing_analyst", "SELECT * FROM customer LIMIT 5")
print("Columns visible to the analyst:", list(df.columns))   # full_name, email, phone are absent
df

## How the governance chain is provisioned in Terraform (IaC)
Why can't `co_marketing_analyst` bypass Athena and read the raw Parquet/CSV files directly from
S3? Inspect the resource declarations in `terraform/governed-data-product/main.tf` to see how
storage, catalog, Lake Formation, and IAM fit together:

In [ ]:
from pathlib import Path

tf_path = Path("terraform/governed-data-product/main.tf")
for line in tf_path.read_text().splitlines():
    if line.startswith(("# ---", "resource ", "variable ", "output ")):
        print(line)

Notice the **governance chain** in `main.tf`: `aws_s3_bucket` (storage) is registered with
`aws_lakeformation_resource`, cataloged by `aws_glue_catalog_table`, and granted to
`aws_iam_role.consumer` via `aws_lakeformation_permissions` — while the consumer IAM policy grants
`athena:*` and `lakeformation:GetDataAccess` with **no `s3:GetObject` on the data bucket**.

**Takeaway / your turn:** that's one persona and a preview of the IaC module. In the exercise
you'll configure all four personas — column, row, tag-based, and metadata-only — verify them
with `verify_step7(gc)`, and answer the architecture questions on `main.tf`.